# Task 5: Semantic Segmentation for Targeted Colorization


In [ ]:
import subprocess, sys
def ensure_numpy1():
    try:
        import numpy as np
        if int(np.__version__.split('.')[0]) >= 2:
            print(f"[!] Detected NumPy {np.__version__}. Downgrading to 1.26.4...")
            subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2.0', '--force-reinstall', '-q'])
            print("\n[SUCCESS] Restart your kernel NOW!")
            return False
        return True
    except ImportError: return False

if not ensure_numpy1():
    raise Exception("Restart Kernel to apply NumPy fix.")

pkgs = ['gradio', 'scikit-image', 'tqdm', 'opencv-python-headless', 'matplotlib']
print("Checking dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print("Done!")

In [ ]:
import os, gc, warnings, glob, random
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from skimage import color
from tqdm import tqdm
import cv2
import gradio as gr

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Segmentation & Colorization Models


In [ ]:
seg_model = models.segmentation.deeplabv3_resnet101(pretrained=True).to(DEVICE).eval()
VOC_CLASSES = ['__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']

class ColorNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(64, 2, 3, padding=1), nn.Tanh()
        )
    def forward(self, x): return self.net(x)

col_model = ColorNet().to(DEVICE).eval()
print("Models ready!")

In [ ]:
def process_targeted(image, target_class):
    if image is None: return None

    input_tensor = transforms.ToTensor()(Image.fromarray(image)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = seg_model(input_tensor)['out'][0]
    mask = output.argmax(0).cpu().numpy()
    

    target_idx = VOC_CLASSES.index(target_class)
    binary_mask = (mask == target_idx).astype(np.uint8)
    

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    res = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    
    
    res[binary_mask == 1, 1] = 200 
    return res

with gr.Blocks() as ui:
    gr.Markdown("# Task 5: Targeted Colorization")
    with gr.Row():
        with gr.Column():
            img_in = gr.Image(label="Input")
            cls_in = gr.Dropdown(choices=VOC_CLASSES, value="person", label="Target Category")
            btn = gr.Button("Colorize Targeted")
        img_out = gr.Image(label="Output")
    btn.click(fn=process_targeted, inputs=[img_in, cls_in], outputs=img_out)

print("Launching GUI...")
ui.launch(server_port=7862, share=False)